### Part (a)

In [1]:
from pathlib import Path
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import softplus
from scipy.special import ndtr
from sklearn.model_selection import train_test_split
from IPython.display import display


In [2]:
def mean_and_std(output):
    mu = output[:, 0]
    sigma = softplus(output[:, 1])
    return mu, sigma


### Part (b)

In [3]:
def gaussian_loss(output, y):
    mu = output[:, 0]
    sigma = softplus(output[:, 1])
    return (torch.log(sigma) + (y - mu) ** 2 / (2 * sigma ** 2) + 0.5 * np.log(2 * np.pi)).mean()


### Part (c)

In [4]:
torch.manual_seed(42)
np.random.seed(42)
folder = Path('.')
data = pd.read_parquet(folder / 'taxi_trips.parquet')
features = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'min_of_day', 'day_of_week', 'day_of_year']
x = data[features].to_numpy(dtype=np.float32)
y = data['trip_duration'].to_numpy(dtype=np.float64)
display(pd.DataFrame({"rows": [len(data)], "columns": [data.shape[1]], "features": [x.shape[1]]}))


,rows,columns,features
0,523554,8,7


In [5]:
(train, holdout) = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42)
(val, test) = train_test_split(holdout, test_size=0.5, random_state=42)
center = y[train].mean()
scale = y[train].std()
x_train = torch.tensor(x[train])
x_val = torch.tensor(x[val])
y_train = torch.tensor((y[train] - center) / scale, dtype=torch.float32)
y_val = torch.tensor((y[val] - center) / scale, dtype=torch.float32)
pd.DataFrame({"split": ["train", "validation", "test"], "rows": [len(train), len(val), len(test)]})


,split,rows
0,train,366487
1,validation,78533
2,test,78534


In [6]:
model = nn.Sequential(nn.Linear(7, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
best_loss = float('inf')
wait = 0
history = []


In [7]:
for epoch in range(100):
    model.train()
    order = torch.randperm(len(train))
    for start in range(0, len(train), 1024):
        batch = order[start:start + 1024]
        optimizer.zero_grad()
        loss = gaussian_loss(model(x_train[batch]), y_train[batch])
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = gaussian_loss(model(x_val), y_val).item()
    history.append({'epoch': epoch + 1, 'val_nll': val_loss + np.log(scale)})
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = deepcopy(model.state_dict())
        best_epoch = epoch + 1
        wait = 0
    else:
        wait += 1
    if wait == 10:
        break
model.load_state_dict(best_state)
model.eval()
pd.DataFrame([{"selected_epoch": best_epoch, "validation_nll": best_loss + np.log(scale), "baseline_mean_minutes": center, "baseline_std_minutes": scale}])


,selected_epoch,validation_nll,baseline_mean_minutes,baseline_std_minutes
0,48,2.630229,10.416308,7.755263


In [8]:
with torch.no_grad():
    validation_output = model(x_val)
    validation_mu = validation_output[:, 0].numpy().astype(np.float64) * scale + center
baseline_validation_mse = np.mean((y[val] - center) ** 2)
validation_results = pd.DataFrame({'model': ['Constant Gaussian', 'MLP Gaussian'], 'nll': [np.log(scale) + baseline_validation_mse / (2 * scale ** 2) + 0.5 * np.log(2 * np.pi), best_loss + np.log(scale)], 'rmse': [np.sqrt(baseline_validation_mse), np.sqrt(np.mean((y[val] - validation_mu) ** 2))]})
validation_results.to_csv(folder / 'validation_results.csv', index=False)
validation_results


,model,nll,rmse
0,Constant Gaussian,3.466067,7.745618
1,MLP Gaussian,2.630229,3.904418


### Part (d)

In [9]:
with torch.no_grad():
    output = model(torch.tensor(x[test]))
    mu_z, sigma_z = mean_and_std(output)
    mu = mu_z.numpy().astype(np.float64) * scale + center
    sigma = sigma_z.numpy().astype(np.float64) * scale
probability = ndtr((45 - mu) / sigma)
pd.DataFrame({"mean_minutes": mu, "std_minutes": sigma, "probability_below_45": probability}).head()


,mean_minutes,std_minutes,probability_below_45
0,4.345385,3.528594,1.0
1,6.882735,3.379159,1.0
2,11.899460,4.173916,1.0
3,9.207955,2.494851,1.0
4,5.875722,3.464281,1.0


### Part (e)

In [10]:
def evaluate(name, y, mu, sigma):
    probability = ndtr((45 - mu) / sigma)
    event = y < 45
    return {'model': name, 'nll': np.mean(np.log(sigma) + (y - mu) ** 2 / (2 * sigma ** 2) + 0.5 * np.log(2 * np.pi)), 'mse': np.mean((y - mu) ** 2), 'brier': np.mean((probability - event) ** 2), 'accuracy': np.mean((probability >= 0.5) == event)}


In [11]:
results = pd.DataFrame([evaluate('Constant Gaussian', y[test], center, scale), evaluate('MLP Gaussian', y[test], mu, sigma)])
results.to_csv(folder / 'results.csv', index=False)
pd.DataFrame(history).to_csv(folder / 'training.csv', index=False)
pd.DataFrame([{'seed': 42, 'train_n': len(train), 'val_n': len(val), 'test_n': len(test), 'target_center': center, 'target_scale': scale, 'best_epoch': best_epoch, 'test_fraction_below_45': np.mean(y[test] < 45), 'test_min_predicted_sigma': sigma.min()}]).to_csv(folder / 'run_details.csv', index=False)
test_performance = results.assign(rmse=np.sqrt(results["mse"]))[["model", "nll", "mse", "rmse", "brier", "accuracy"]]
test_performance


,model,nll,mse,rmse,brier,accuracy
0,Constant Gaussian,3.467520,60.169377,7.756892,0.004457,0.995543
1,MLP Gaussian,2.756783,16.163363,4.020369,0.002925,0.996677
